# Кредитный скоринг: обучение и экспорт

## 1. Библиотеки и пути

In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if root is None:
    raise FileNotFoundError("Запустите ноутбук из директории проекта или notebooks/")
path = root / "datasets/german_credit_data.csv"
if not path.is_file():
    raise FileNotFoundError(f"Нет {path}. Инструкция по данным: {root / 'datasets/README.md'}")


## 2. Признаки

Соответствие столбцов задано по заголовку CSV. Индекс строки и целевой столбец `Risk` не используются как признаки. Job обрабатывается как категория; числовые age, credit_amount и duration масштабируются.

In [2]:
SOURCE_URL = "https://www.kaggle.com/datasets/kabure/german-credit-data-with-risk"
# Имена справа — публичный контракт API; имена слева взяты из заголовка CSV.
COLUMNS = {
    "Age": "age",
    "Sex": "sex",
    "Job": "job",
    "Housing": "housing",
    "Saving accounts": "saving_accounts",
    "Checking account": "checking_account",
    "Credit amount": "credit_amount",
    "Duration": "duration",
    "Purpose": "purpose",
}
FEATURES = list(COLUMNS.values())
NUMERIC = ["age", "credit_amount", "duration"]
CATEGORICAL = [name for name in FEATURES if name not in NUMERIC]

## 3. Загрузка и проверка данных

Цель: `bad` → 1, `good` → 0. pandas читает маркер NA как пропуск; ниже выводим фактические размеры, классы и пропуски, а не задаём их числа вручную.

In [3]:
data = pd.read_csv(path)
if set(data["Risk"].unique()) != {"good", "bad"}:
    raise ValueError("Expected Risk labels good/bad")
frame = data[list(COLUMNS)].rename(columns=COLUMNS)
target = data["Risk"].eq("bad").astype(int)
print("Размер CSV:", data.shape)
print("Классы:", data["Risk"].value_counts().to_dict())
print("Пропуски:", frame.isna().sum().to_dict())
frame.head()

Размер CSV: (1000, 11)
Классы: {'good': 700, 'bad': 300}
Пропуски: {'age': 0, 'sex': 0, 'job': 0, 'housing': 0, 'saving_accounts': 183, 'checking_account': 394, 'credit_amount': 0, 'duration': 0, 'purpose': 0}


,age,sex,job,housing,saving_accounts,checking_account,credit_amount,duration,purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


## 4. Разделение выборки

80/20, stratify и random_state=42

In [4]:
x_train, x_test, y_train, y_test = train_test_split(frame, target, test_size=0.2, random_state=42, stratify=target)
print("Train:", len(x_train), "Test:", len(x_test))

Train: 800 Test: 200


## 5. Препроцессинг и классификатор

Числа: StandardScaler. Категории: SimpleImputer с отдельной категорией missing и OneHotEncoder. Финальный шаг — LogisticRegression. Pipeline объединяет преобразования и модель, чтобы при инференсе применялась та же обработка.

In [5]:
categorical = Pipeline(
    [
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocess = ColumnTransformer(
    [
        ("numeric", StandardScaler(), NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ]
)
pipeline = Pipeline([("preprocess", preprocess), ("model", LogisticRegression(max_iter=1000, random_state=42))])

## 6. Обучение

Обучаем Pipeline только на тренировочной части.

In [ ]:
pipeline.fit(x_train, y_train)

## 7. Оценка на отложенной части

score — вероятность класса 1 (bad). Порог 0.5

In [7]:
scores = pipeline.predict_proba(x_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, scores >= 0.5))
print("ROC-AUC:", roc_auc_score(y_test, scores))

Accuracy: 0.74
ROC-AUC: 0.7613095238095238


## 8. Паспорт модели

Фиксируем порядок признаков, версию, порог, источник и SHA-256 CSV. Категории, диапазоны и пропуски вычисляются по исходным данным для описания контракта; метрики — только по test.

In [ ]:
# 0.5 — выбранный порог
metadata = {
    "model_version": "german-credit-1.0",
    "features": FEATURES,
    "threshold": 0.5,
    "positive_class": "bad",
    "negative_class": "good",
    "target_column": "Risk",
    "source_url": SOURCE_URL,
    "source_file": path.name,
    "source_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    "sklearn_version": sklearn.__version__,
    "random_state": 42,
    "train_rows": len(x_train),
    "test_rows": len(x_test),
    "class_counts": data["Risk"].value_counts().to_dict(),
    "missing_values": frame.isna().sum().to_dict(),
    "observed_categories": {name: sorted(frame[name].dropna().unique().tolist()) for name in CATEGORICAL},
    "observed_numeric_ranges": {name: {"min": float(frame[name].min()), "max": float(frame[name].max())} for name in NUMERIC},
    "test_accuracy": float(accuracy_score(y_test, scores >= 0.5)),
    "test_roc_auc": float(roc_auc_score(y_test, scores)),
}
metadata

{'model_version': 'german-credit-1.0',
 'features': ['age',
  'sex',
  'job',
  'housing',
  'saving_accounts',
  'checking_account',
  'credit_amount',
  'duration',
  'purpose'],
 'threshold': 0.5,
 'positive_class': 'bad',
 'negative_class': 'good',
 'target_column': 'Risk',
 'source_url': 'https://www.kaggle.com/datasets/kabure/german-credit-data-with-risk',
 'source_file': 'german_credit_data.csv',
 'source_sha256': '42be3b82a2e5073bd5ca23bce1d1c31426b78f72d20cd892f3aacaa2ba30a075',
 'sklearn_version': '1.6.1',
 'random_state': 42,
 'train_rows': 800,
 'test_rows': 200,
 'class_counts': {'good': 700, 'bad': 300},
 'missing_values': {'age': 0,
  'sex': 0,
  'job': 0,
  'housing': 0,
  'saving_accounts': 183,
  'checking_account': 394,
  'credit_amount': 0,
  'duration': 0,
  'purpose': 0},
 'observed_categories': {'sex': ['female', 'male'],
  'job': [0, 1, 2, 3],
  'housing': ['free', 'own', 'rent'],
  'saving_accounts': ['little', 'moderate', 'quite rich', 'rich'],
  'checking_acc

## 9. Экспорт для сервиса

Бандл содержит два ключа: pipeline и metadata. Отдельный JSON — читаемая копия паспорта; сервис использует паспорт внутри joblib. После обновления артефакта перезапустите API, а для Docker пересоберите образ.

In [ ]:
folder = root / "artifact"
folder.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipeline, "metadata": metadata}, folder / "model.joblib")
(folder / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
(folder / "dataset_description.txt").write_text(
    "German Credit Risk - With Target\n" + SOURCE_URL + "\n"
    "Source CSV: datasets/german_credit_data.csv; target: Risk (bad=1, good=0).\n"
    "Column mapping and missing values: datasets/README.md and metadata.json.\n"
)

## 10. Проверка сохранённого Pipeline

Повторно загружаем joblib, проверяем порядок признаков и совпадение вероятностей с моделью до сохранения. Импортов из credit_service для обучения и загрузки нет.

In [10]:
import numpy as np

bundle = joblib.load(folder / "model.joblib")
assert set(bundle) == {"pipeline", "metadata"}
assert bundle["metadata"]["features"] == list(x_test.columns)
np.testing.assert_allclose(bundle["pipeline"].predict_proba(x_test)[:, 1], scores, rtol=0, atol=0)
print("Загруженный Pipeline воспроизводит предсказания; артефакт готов для сервиса")

Загруженный Pipeline воспроизводит предсказания; артефакт готов для сервиса
